bài 1 SIMPLE REFLEX AGENT CHƠI 8PUZZLE THEO LUẬT

In [3]:
import random
class Puzzle8:
    def __init__(self, board):
        # board là mảng 2 chiều, ví dụ: [[1, 3, 5],
        #                                [4, 6, 8],
        #                                [2, 7, 0]]
        #ô trống biểu diễn là số 0
        self.board = board
        self.rows = 3
        self.cols = 3
        # Tìm vị trí ô trống (số 0)
        self.blank_pos = self._find_blank()

    def _find_blank(self):
        for r in range(self.rows):
            for c in range(self.cols):
                if self.board[r][c] == 0:
                    return (r, c)

    def get_valid_rules(self):
        """Trả về danh sách các hướng ô trống có thể di chuyển"""
        rules = []
        r, c = self.blank_pos
        if r > 0: rules.append("UP")
        if r < 2: rules.append("DOWN")
        if c > 0: rules.append("LEFT")
        if c < 2: rules.append("RIGHT")
        return rules

    def move(self, direction):
        """Thực hiện di chuyển và trả về trạng thái mới"""
        r, c = self.blank_pos
        new_board = [row[:] for row in self.board]  # Copy bảng hiện tại

        if direction == "UP":
            new_board[r][c], new_board[r - 1][c] = new_board[r - 1][c], new_board[r][c]
        elif direction == "DOWN":
            new_board[r][c], new_board[r + 1][c] = new_board[r + 1][c], new_board[r][c]
        elif direction == "LEFT":
            new_board[r][c], new_board[r][c - 1] = new_board[r][c - 1], new_board[r][c]
        elif direction == "RIGHT":
            new_board[r][c], new_board[r][c + 1] = new_board[r][c + 1], new_board[r][c]
        return Puzzle8(new_board)

    def display(self):
        for row in self.board:
            print(row)
        print("---")


# Khởi tạo trạng thái ví dụ
current_board = [
    [1, 3, 5],
    [4, 6, 8],
    [2, 7, 0] # 0 đại diện cho ô trống ở góc dưới phải
]

puzzle = Puzzle8(current_board)

# Kiểm tra các quy tắc hợp lệ
print("Các hướng có thể đi:", puzzle.get_valid_rules())

# Thử thực hiện một bước ngẫu nhiên dựa trên tập rule
rule_action = random.choice(puzzle.get_valid_rules())
print(f"Đang di chuyển ô trống sang {rule_action}...")
new_state = puzzle.move(rule_action)
new_state.display()


bài 2:based reflex agent hoàn thành trò chơi đề ngẫu nhiên

In [11]:

from collections import deque
import random
import copy


# ──────────────────────────────────────────────
# Cấu trúc dữ liệu
# ──────────────────────────────────────────────

GOAL_STATE = (1, 2, 3, 4, 5, 6, 7, 8, 0)   # 0 = ô trống

DIRECTIONS = {
    'UP':    -3,
    'DOWN':  +3,
    'LEFT':  -1,
    'RIGHT': +1,
}


class Model:
    """
    Model lưu trữ kiến thức của agent về thế giới.
    Bao gồm: trạng thái đã biết và hành động vừa thực hiện.
    """
    def __init__(self, initial_state):
        self.known_state = initial_state
        self.last_action = None

    def __repr__(self):
        return f"Model(known_state={self.known_state}, last_action={self.last_action})"


class Rule:
    """
    Một quy tắc: nếu điều kiện thoả mãn thì thực hiện action.
    """
    def __init__(self, condition_fn, action: str):
        self.condition_fn = condition_fn   # f(state) -> bool
        self.action = action

    def matches(self, state) -> bool:
        return self.condition_fn(state)


# ──────────────────────────────────────────────
# Hàm tiện ích cho 8-puzzle
# ──────────────────────────────────────────────

def find_blank(state: tuple) -> int:
    """Trả về chỉ số của ô trống (0)."""
    return state.index(0)


def get_valid_moves(state: tuple) -> list[tuple[str, int]]:
    """
    Trả về danh sách các nước đi hợp lệ dạng (direction, blank_new_idx).
    """
    blank = find_blank(state)
    row, col = divmod(blank, 3)
    moves = []
    if row > 0: moves.append(('UP',    blank - 3))
    if row < 2: moves.append(('DOWN',  blank + 3))
    if col > 0: moves.append(('LEFT',  blank - 1))
    if col < 2: moves.append(('RIGHT', blank + 1))
    return moves


def apply_action(state: tuple, action: str) -> tuple:
    """Áp dụng một hành động và trả về trạng thái mới."""
    blank = find_blank(state)
    delta = DIRECTIONS[action]
    new_blank = blank + delta
    lst = list(state)
    lst[blank], lst[new_blank] = lst[new_blank], lst[blank]
    return tuple(lst)


def is_goal(state: tuple) -> bool:
    return state == GOAL_STATE


def misplaced_tiles(state: tuple) -> int:
    """Heuristic: số ô sai vị trí (không tính ô trống)."""
    return sum(1 for i, v in enumerate(state) if v != 0 and v != GOAL_STATE[i])


def is_solvable(state: tuple) -> bool:
    """Kiểm tra puzzle có giải được không (số nghịch thế chẵn)."""
    tiles = [v for v in state if v != 0]
    inversions = sum(
        1 for i in range(len(tiles))
          for j in range(i + 1, len(tiles))
          if tiles[i] > tiles[j]
    )
    return inversions % 2 == 0


def print_board(state: tuple, label: str = ""):
    if label:
        print(f"\n{label}")
    for i in range(0, 9, 3):
        row = state[i:i+3]
        print("  " + " ".join(str(v) if v != 0 else '.' for v in row))


# ──────────────────────────────────────────────
# BFS xây dựng tập quy tắc (rules)
# BFS tìm đường đi tối ưu từ state -> goal
# ──────────────────────────────────────────────

def bfs_solve(initial_state: tuple) -> list[str] | None:
    """
    Tìm dãy action tối ưu bằng BFS.
    Trả về list[action] hoặc None nếu không giải được.
    """
    if is_goal(initial_state):
        return []
    queue = deque([(initial_state, [])])
    visited = {initial_state}
    while queue:
        state, path = queue.popleft()
        for direction, new_blank in get_valid_moves(state):
            next_state = apply_action(state, direction)
            if next_state not in visited:
                new_path = path + [direction]
                if is_goal(next_state):
                    return new_path
                visited.add(next_state)
                queue.append((next_state, new_path))
    return None


def build_rules(action_sequence: list[str]) -> list[Rule]:
    """
    Xây dựng danh sách Rule từ chuỗi action tối ưu.
    Mỗi rule ứng với 1 action trong chuỗi (theo thứ tự).
    Rule được thiết kế: điều kiện luôn True vì agent đã biết trạng thái.
    """
    rules = []
    for action in action_sequence:
        rule = Rule(condition_fn=lambda state, a=action: True, action=action)
        rules.append(rule)
    return rules


# ──────────────────────────────────────────────
# Các hàm chính của Model-Based Reflex Agent
# ──────────────────────────────────────────────

def update_state(state: tuple, action: str | None,
                 percept: tuple, model: Model) -> tuple:
    """
    UPDATE-STATE(state, action, percept, model)
    Cập nhật trạng thái dựa trên percept từ môi trường.
    Trong 8-puzzle, percept chính là bàn cờ quan sát được.
    """
    # Cập nhật model
    model.known_state = percept
    model.last_action = action
    # Trả về trạng thái mới = percept (quan sát trực tiếp)
    return percept


def rule_match(state: tuple, rules: list[Rule]) -> Rule | None:
    """
    RULE-MATCH(state, rules)
    Tìm quy tắc đầu tiên khớp với trạng thái hiện tại.
    """
    for rule in rules:
        if rule.matches(state):
            return rule
    return None


def model_based_reflex_agent(initial_state: tuple, verbose: bool = True) -> bool:
    """
    Hàm chính: MODEL-BASED-REFLEX-AGENT

    persistent: state, model(state, action)
    state  <- UPDATE-STATE(state, action, percept, model)
    rule   <- RULE-MATCH(state, rules)
    action <- rule.ACTION
    return action
    """
    print("=" * 50)
    print("  MODEL-BASED REFLEX AGENT — 8 Puzzle")
    print("=" * 50)
    print_board(initial_state, "Trạng thái ban đầu:")
    print_board(GOAL_STATE,    "Trạng thái đích:")

    if not is_solvable(initial_state):
        print("\n[!] Puzzle này không có lời giải.")
        return False

    # ── Bước 1: BFS tìm solution path → xây rules ──
    action_sequence = bfs_solve(initial_state)
    if action_sequence is None:
        print("\n[!] Không tìm được lời giải.")
        return False

    rules = build_rules(action_sequence)
    print(f"\n[BFS] Tìm thấy lời giải tối ưu: {len(action_sequence)} bước")
    print(f"      Dãy hành động: {' → '.join(action_sequence)}\n")

    # ── Persistent state ──
    state  = initial_state
    model  = Model(initial_state)
    action = None          # hành động vừa thực hiện (None ở bước đầu)
    step   = 0

    print("-" * 50)
    print(f"{'Bước':<6} {'Action':<8} {'Heuristic (h)':<15} {'Trạng thái'}")
    print("-" * 50)

    while not is_goal(state):
        # Nhận percept từ môi trường (= state hiện tại)
        percept = state

        # UPDATE-STATE
        state = update_state(state, action, percept, model)

        # RULE-MATCH
        rule = rule_match(state, rules)
        if rule is None:
            print("[!] Không tìm thấy rule phù hợp. Dừng lại.")
            return False

        # action <- rule.ACTION
        action = rule.action
        rules.remove(rule)         # dùng xong thì bỏ

        # Thực thi action lên môi trường
        state = apply_action(state, action)
        step += 1
        h = misplaced_tiles(state)

        if verbose:
            board_str = str(state)
            print(f"{step:<6} {action:<8} {h:<15} {board_str}")

    print("-" * 50)
    print(f"\n✓ Hoàn thành! Giải trong {step} bước.\n")
    print_board(state, "Trạng thái cuối:")
    return True


# ──────────────────────────────────────────────
# Tạo puzzle ngẫu nhiên có lời giải
# ──────────────────────────────────────────────

def random_puzzle(num_shuffles: int = 30) -> tuple:
    """Tạo puzzle bằng cách shuffle ngẫu nhiên từ goal state."""
    state = list(GOAL_STATE)
    for _ in range(num_shuffles):
        blank = state.index(0)
        row, col = divmod(blank, 3)
        choices = []
        if row > 0: choices.append(blank - 3)
        if row < 2: choices.append(blank + 3)
        if col > 0: choices.append(blank - 1)
        if col < 2: choices.append(blank + 1)
        swap = random.choice(choices)
        state[blank], state[swap] = state[swap], state[blank]
    return tuple(state)


# ──────────────────────────────────────────────
# Chạy thử
# ──────────────────────────────────────────────

if __name__ == "__main__":
  # Tạo puzzle ngẫu nhiên
  example = random_puzzle(num_shuffles=20)

  # Chạy agent
  model_based_reflex_agent(example, verbose=True)

  MODEL-BASED REFLEX AGENT — 8 Puzzle

Trạng thái ban đầu:
  1 3 6
  4 . 2
  7 5 8

Trạng thái đích:
  1 2 3
  4 5 6
  7 8 .

[BFS] Tìm thấy lời giải tối ưu: 6 bước
      Dãy hành động: RIGHT → UP → LEFT → DOWN → DOWN → RIGHT

--------------------------------------------------
Bước   Action   Heuristic (h)   Trạng thái
--------------------------------------------------
1      RIGHT    5               (1, 3, 6, 4, 2, 0, 7, 5, 8)
2      UP       4               (1, 3, 0, 4, 2, 6, 7, 5, 8)
3      LEFT     3               (1, 0, 3, 4, 2, 6, 7, 5, 8)
4      DOWN     2               (1, 2, 3, 4, 0, 6, 7, 5, 8)
5      DOWN     1               (1, 2, 3, 4, 5, 6, 7, 0, 8)
6      RIGHT    0               (1, 2, 3, 4, 5, 6, 7, 8, 0)
--------------------------------------------------

✓ Hoàn thành! Giải trong 6 bước.


Trạng thái cuối:
  1 2 3
  4 5 6
  7 8 .
